# VerdaSense — RAGAS Ablation Notebook (v2, updated)

**KEY CHANGES:**
- Uses v2 testset (`wound_testset_v2.json`) with `time_payload` field
- Sends `time_payload` directly to FastAPI endpoint (not user_input narrative as notes)
- Adds rule-based safety checker alongside RAGAS metrics
- Saves per-case safety report CSV

In [ ]:
# wound_ragas_ablation_v2.ipynb  (converted to .py for reference)
# ═══════════════════════════════════════════════════════════════════════════
# KEY CHANGES vs original wound_ragas_ablation.ipynb:
#
#  1. READS v2 TESTSET (wound_testset_v2.json) with new field structure:
#       time_payload  (structured inputs — sent to API exactly)
#       user_input    (formatted string — stored in RAGAS SingleTurnSample)
#       reference     (ground truth answer)
#       reference_contexts  (from KNOWLEDGE_BASE_final.json page_content)
#       wound_type_expected, allowed_dressings, contraindicated_dressings,
#       antibiotic_required, referral_required  (for rule-based checks)
#
#  2. SENDS time_payload FIELDS directly to the FastAPI endpoint
#     (not the old user_input narrative as notes)
#
#  3. ADDS RULE-BASED SAFETY CHECKER alongside RAGAS metrics:
#       - contraindicated dressings not in answer
#       - antibiotic mentioned when required
#       - referral mentioned when required
#       - at least one allowed dressing in answer
#
#  4. SAVES SEPARATE SAFETY REPORT alongside the ablation results JSON
# ═══════════════════════════════════════════════════════════════════════════

# ──────────────────────────────────────────────────────────────────────────────
# Cell 1: Imports
# ──────────────────────────────────────────────────────────────────────────────
import os
import json
import time
import ast
import pandas as pd
import httpx
from dotenv import load_dotenv

load_dotenv()

from ragas import evaluate, EvaluationDataset, SingleTurnSample
from ragas.metrics import (
    LLMContextPrecisionWithReference,
    LLMContextRecall,
    Faithfulness,
    AnswerRelevancy,
)
from ragas.llms       import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    import numpy as np
    HAS_MPL = True
except ImportError:
    HAS_MPL = False


# ──────────────────────────────────────────────────────────────────────────────
# Cell 2: RAGAS judge setup
# ──────────────────────────────────────────────────────────────────────────────
judge_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0))
judge_emb = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))


# ──────────────────────────────────────────────────────────────────────────────
# Cell 3: Load v2 testset
# ──────────────────────────────────────────────────────────────────────────────
OUTPUT_DIR   = "../ragas_testset/"
TESTSET_JSON = os.path.join(OUTPUT_DIR, "wound_testset_v2.json")

# Fall back to v1 testset if v2 not found yet
if not os.path.isfile(TESTSET_JSON):
    TESTSET_JSON = os.path.join(OUTPUT_DIR, "wound_testset_curated.json")
    print(f"⚠️  wound_testset_v2.json not found — falling back to wound_testset_curated.json")
    print(f"   Build v2 testset first for accurate evaluation!")

with open(TESTSET_JSON, "r", encoding="utf-8") as f:
    testset = json.load(f)

print(f"✅ Loaded {len(testset)} test cases from {os.path.basename(TESTSET_JSON)}")

# ── Validate testset structure ─────────────────────────────────────────────────
v2_fields = ["time_payload", "reference_contexts", "wound_type_expected",
             "allowed_dressings", "contraindicated_dressings"]
has_v2 = all(f in testset[0] for f in v2_fields) if testset else False

if has_v2:
    print("   ✅ v2 testset format detected — full evaluation enabled")
    cats = {}
    for tc in testset:
        c = tc.get("category", "?")
        cats[c] = cats.get(c, 0) + 1
    for c, n in sorted(cats.items()):
        print(f"      Category {c}: {n} cases")
else:
    print("   ⚠️  v1 testset format detected — rule-based safety checks will be skipped")
    print("      Rebuild testset using wound_testset_v2_scaffold.json for full evaluation")


# ──────────────────────────────────────────────────────────────────────────────
# Cell 4: RAG caller — UPDATED to use time_payload
# ──────────────────────────────────────────────────────────────────────────────
RAG_URL = "http://localhost:8000/get_recommendation"

def call_rag(record: dict, timeout: int = 120) -> dict:
    """
    Send T.I.M.E. inputs to the running FastAPI server.

    CHANGE vs v1: uses time_payload dict for all fields.
    The 'notes' field comes from time_payload["notes"] (empty string if not set).
    user_input is NOT sent as notes — it's stored in the RAGAS sample separately.
    """
    if has_v2:
        # v2 testset: use time_payload directly
        tp = record["time_payload"]
        payload = {
            "necrotic_pct":    tp["necrotic_pct"],
            "slough_pct":      tp["slough_pct"],
            "granulation_pct": tp["granulation_pct"],
            "infection":       tp["infection"],
            "moisture":        tp["moisture"],
            "edge":            tp["edge"],
            "notes":           tp.get("notes", ""),
            "tissue_confidence": 0.0,
        }
    else:
        # v1 testset: use time_inputs (legacy)
        t = record.get("time_inputs", {})
        payload = {
            "necrotic_pct":    t.get("necrotic_pct", 0),
            "slough_pct":      t.get("slough_pct", 0),
            "granulation_pct": t.get("granulation_pct", 100),
            "infection":       t.get("infection", "Not infected"),
            "moisture":        t.get("moisture", "Low"),
            "edge":            t.get("edge", "Advancing"),
            "notes":           record.get("user_input", ""),
            "tissue_confidence": 0.0,
        }

    try:
        r = httpx.post(RAG_URL, data=payload, timeout=timeout)
        r.raise_for_status()
        return r.json()
    except Exception as e:
        return {"result": f"ERROR: {e}", "chunk_texts": [], "confidence_label": "LOW"}


# ──────────────────────────────────────────────────────────────────────────────
# Cell 5: Rule-based safety checker (NEW)
# ──────────────────────────────────────────────────────────────────────────────
def check_safety(generated_answer: str, test_case: dict) -> dict:
    """
    Rule-based safety evaluation — catches clinical errors that RAGAS misses.

    Checks:
    1. Contraindicated dressings NOT in the answer
    2. Antibiotic recommendation present when required
    3. Referral recommendation present when required
    4. At least one allowed dressing mentioned in answer

    Returns dict with per-check PASS/FAIL and overall result.
    """
    if not has_v2:
        return {}

    ans_lower = generated_answer.lower()
    results   = {}

    # 1. Contraindicated dressings
    contraindicated = test_case.get("contraindicated_dressings", [])
    for contra in contraindicated:
        found = contra.lower() in ans_lower
        results[f"contraindication_absent_{contra}"] = "FAIL" if found else "PASS"

    # 2. Antibiotic mentioned when required
    if test_case.get("antibiotic_required", False):
        antibiotic_mentioned = any(
            kw in ans_lower for kw in
            ["antibiotic", "c&s", "culture and sensitivity", "systemic", "antimicrobial therapy"]
        )
        results["antibiotic_recommended"] = "PASS" if antibiotic_mentioned else "FAIL"

    # 3. Referral mentioned when required
    if test_case.get("referral_required", False):
        referral_mentioned = any(
            kw in ans_lower for kw in
            ["refer", "hospital", "specialist", "escalat", "wound type 6",
             "wound type 7", "wound type 8"]
        )
        results["referral_recommended"] = "PASS" if referral_mentioned else "FAIL"

    # 4. Allowed dressings
    allowed = test_case.get("allowed_dressings", [])
    if allowed:
        any_allowed = any(d.lower() in ans_lower for d in allowed)
        results["dressing_in_allowed_list"] = "PASS" if any_allowed else "FAIL"

    overall = "PASS" if all(v == "PASS" for v in results.values()) else "FAIL"
    results["overall"] = overall
    return results


# ──────────────────────────────────────────────────────────────────────────────
# Cell 6: Metric column definitions
# ──────────────────────────────────────────────────────────────────────────────
METRIC_COLS = [
    "llm_context_precision_with_reference",
    "context_recall",
    "faithfulness",
    "answer_relevancy",
]
METRIC_LABELS = [
    "Context precision",
    "Context recall",
    "Faithfulness",
    "Answer relevancy",
]
METRIC_COLORS = ["#185FA5", "#1D9E75", "#BA7517", "#D85A30"]


# ──────────────────────────────────────────────────────────────────────────────
# Cell 7: Main evaluation function — UPDATED
# ──────────────────────────────────────────────────────────────────────────────
def run_evaluation(experiment_name: str, results_json: str):
    """
    Run RAGAS + rule-based safety evaluation for one experiment.

    Steps:
    1. Call RAG server for each test case using time_payload
    2. Apply rule-based safety checker on each generated answer
    3. Score with RAGAS metrics
    4. Save results JSON + safety report CSV
    """
    print(f"\n{'='*60}")
    print(f"EXPERIMENT: {experiment_name}")
    print(f"{'='*60}")

    safety_report_path = results_json.replace(".json", "_safety.csv")

    if os.path.isfile(results_json):
        with open(results_json, encoding="utf-8") as f:
            records = json.load(f)
        done = {r["index"] for r in records}
        print(f"Resuming: {len(records)}/{len(testset)} done")
    else:
        records = []
        done    = set()

    for idx, case in enumerate(testset):
        if idx in done:
            print(f"  [{idx+1:>2}/{len(testset)}] skip (done)")
            continue

        name = case.get("synthesizer_name", case.get("case_id", f"case_{idx}"))
        print(f"  [{idx+1:>2}/{len(testset)}] {name}")
        t0      = time.time()
        resp    = call_rag(case)
        elapsed = time.time() - t0

        answer = resp.get("result", "")
        chunks = resp.get("chunk_texts", [])
        retrieved_contexts = chunks if chunks else [answer]

        # ── Rule-based safety check ────────────────────────────────────────────
        safety = check_safety(answer, case)
        if safety:
            status = safety.get("overall", "N/A")
            print(f"       Safety: {status}")

        # ── Determine user_input for RAGAS ────────────────────────────────────
        # v2: use the formatted user_input string (already in API format)
        # v1: use the narrative user_input
        user_input_for_ragas = case.get("user_input", "")

        records.append({
            "index":               idx,
            "case_id":             case.get("case_id", case.get("synthesizer_name", f"case_{idx}")),
            "category":            case.get("category", "?"),
            "synthesizer_name":    name,
            "wound_type_expected": case.get("wound_type_expected", "?"),
            "user_input":          user_input_for_ragas,
            "reference":           case.get("reference", ""),
            "reference_contexts":  case.get("reference_contexts", []),
            "retrieved_contexts":  retrieved_contexts,
            "answer":              answer,
            "confidence_label":    resp.get("confidence_label", "?"),
            "elapsed_sec":         round(elapsed, 1),
            "safety_checks":       safety,
        })

        with open(results_json, "w", encoding="utf-8") as f:
            json.dump(records, f, indent=2, ensure_ascii=False)

        time.sleep(1.0)

    # ── Save safety report ─────────────────────────────────────────────────────
    if has_v2:
        safety_rows = []
        for r in records:
            row = {
                "case_id":             r.get("case_id", ""),
                "category":            r.get("category", "?"),
                "wound_type_expected": r.get("wound_type_expected", "?"),
                "overall":             r.get("safety_checks", {}).get("overall", "N/A"),
            }
            for k, v in r.get("safety_checks", {}).items():
                if k != "overall":
                    row[k] = v
            safety_rows.append(row)

        safety_df = pd.DataFrame(safety_rows)
        safety_df.to_csv(safety_report_path, index=False)

        total = len(safety_df)
        passed = (safety_df["overall"] == "PASS").sum() if "overall" in safety_df.columns else 0
        print(f"\n🛡️  Safety Report: {passed}/{total} cases PASS ({100*passed//total if total else 0}%)")
        print(f"   Saved → {safety_report_path}")

        if "overall" in safety_df.columns:
            failed = safety_df[safety_df["overall"] == "FAIL"]
            if not failed.empty:
                print(f"\n   ❌ FAILED cases ({len(failed)}):")
                for _, row in failed.iterrows():
                    checks = [k for k, v in row.items()
                              if k not in ("case_id","category","wound_type_expected","overall")
                              and v == "FAIL"]
                    print(f"      {row['case_id']} — {', '.join(checks)}")

    # ── Build RAGAS dataset ────────────────────────────────────────────────────
    samples = []
    for r in records:
        if not r["answer"] or r["answer"].startswith("ERROR"):
            continue

        ref_contexts = r.get("reference_contexts", [])
        if not ref_contexts:
            print(f"   ⚠️  No reference_contexts for {r.get('case_id', '?')} — RAGAS recall will be 0")

        samples.append(SingleTurnSample(
            user_input         = r["user_input"],
            reference          = r["reference"],
            reference_contexts = [str(c) for c in ref_contexts],
            retrieved_contexts = [str(c) for c in r["retrieved_contexts"]],
            response           = r["answer"],
        ))

    print(f"\nRunning RAGAS scoring on {len(samples)} samples...")
    dataset = EvaluationDataset(samples)
    results = evaluate(dataset, metrics=[
        LLMContextPrecisionWithReference(llm=judge_llm),
        LLMContextRecall(llm=judge_llm),
        Faithfulness(llm=judge_llm),
        AnswerRelevancy(llm=judge_llm, embeddings=judge_emb),
    ])

    scores_df = results.to_pandas()
    for col in METRIC_COLS:
        if col not in scores_df.columns:
            scores_df[col] = float("nan")

    agg = {col: scores_df[col].mean() for col in METRIC_COLS}

    print(f"\n{'─'*40}")
    print(f"  RAGAS SCORES — {experiment_name}")
    print(f"{'─'*40}")
    for label, col in zip(METRIC_LABELS, METRIC_COLS):
        print(f"  {label:<25} {agg[col]:.4f}  ({agg[col]*100:.1f}%)")
    print(f"{'─'*40}")

    # ── Save chart if matplotlib available ────────────────────────────────────
    if HAS_MPL:
        fig, ax = plt.subplots(figsize=(8, 4))
        vals   = [agg[c] for c in METRIC_COLS]
        colors = METRIC_COLORS
        bars   = ax.bar(METRIC_LABELS, vals, color=colors, alpha=0.85, zorder=3)
        ax.set_ylim(0, 1.0)
        ax.set_ylabel("Score")
        ax.set_title(f"RAGAS Metrics — {experiment_name}")
        ax.grid(axis="y", alpha=0.3, zorder=0)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                    f"{val:.3f}", ha="center", va="bottom", fontsize=9)
        chart_path = results_json.replace(".json", "_chart.png")
        fig.tight_layout()
        fig.savefig(chart_path, dpi=150)
        plt.close(fig)
        print(f"   Chart saved → {chart_path}")

    agg_df = pd.DataFrame([{"metric": lbl, "score": agg[col]}
                            for lbl, col in zip(METRIC_LABELS, METRIC_COLS)])
    return agg_df, scores_df

c:\Users\GIGA\OneDrive - Universiti Malaya\Documents\rag-for-beginners\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\GIGA\AppData\Local\Temp\ipykernel_7384\1291281545.py:39: DeprecationWarning: Importing LLMContextPrecisionWithReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextPrecisionWithReference
  from ragas.metrics import (
C:\Users\GIGA\AppData\Local\Temp\ipykernel_7384\1291281545.py:39: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import (
C:\Users\GIGA\AppData\Local\Tem

✅ Loaded 28 test cases from wound_testset_v2.json
   ✅ v2 testset format detected — full evaluation enabled
      Category A: 8 cases
      Category B: 10 cases
      Category C: 6 cases
      Category D: 4 cases


C:\Users\GIGA\AppData\Local\Temp\ipykernel_7384\1291281545.py:64: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  judge_emb = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))


In [3]:
# ──────────────────────────────────────────────────────────────────────────────
# Cell 8: Run evaluations
# ──────────────────────────────────────────────────────────────────────────────
# Change experiment_name and results_json for each run.
# Start the matching wound_app_XX.py server before running.

agg_df_00, full_df_00 = run_evaluation(
    experiment_name = "wound_ragas_eval_00_v2",
    results_json    = "../ragas_eval_00_v2/wound_ragas_ablation_results.json",
)
agg_df_00




EXPERIMENT: wound_ragas_eval_00_v2
  [ 1/28] cat_a_type1_dry
       Safety: FAIL
  [ 2/28] cat_a_type2_wet
       Safety: FAIL
  [ 3/28] cat_a_type3_dry_infected
       Safety: FAIL
  [ 4/28] cat_a_type4_wet_infected
       Safety: FAIL
  [ 5/28] cat_a_type5_dry_necrotic
       Safety: FAIL
  [ 6/28] cat_a_type6_wet_necrotic
       Safety: PASS
  [ 7/28] cat_a_type7_dry_infected_necrotic
       Safety: FAIL
  [ 8/28] cat_a_type8_wet_infected_necrotic
       Safety: FAIL
  [ 9/28] cat_b_iodine_thyroid
       Safety: FAIL
  [10/28] cat_b_silver_clean_granulating
       Safety: FAIL
  [11/28] cat_b_skin_tear_fragile
       Safety: FAIL
  [12/28] cat_b_npwt_necrotic_eschar
       Safety: FAIL
  [13/28] cat_b_alginate_dry_wound
       Safety: FAIL
  [14/28] cat_b_honey_dry_necrotic
       Safety: FAIL
  [15/28] cat_b_postop_clean
       Safety: PASS
  [16/28] cat_b_burns_hand
       Safety: FAIL
  [17/28] cat_b_referral_type6
       Safety: PASS
  [18/28] cat_b_diabetic_foot
       Safety:

Evaluating:  20%|█▉        | 22/112 [01:43<04:44,  3.16s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating:  30%|███       | 34/112 [02:22<03:30,  2.70s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating: 100%|██████████| 112/112 [07:55<00:00,  4.25s/it]



────────────────────────────────────────
  RAGAS SCORES — wound_ragas_eval_00_v2
────────────────────────────────────────
  Context precision         0.8663  (86.6%)
  Context recall            0.6582  (65.8%)
  Faithfulness              0.6126  (61.3%)
  Answer relevancy          0.5732  (57.3%)
────────────────────────────────────────
   Chart saved → ../ragas_eval_00_v2/wound_ragas_ablation_results_chart.png


,metric,score
0,Context precision,0.866349
1,Context recall,0.658214
2,Faithfulness,0.612632
3,Answer relevancy,0.573245


In [4]:
# ──────────────────────────────────────────────────────────────────────────────
# Cell 9: Eval 01
# ──────────────────────────────────────────────────────────────────────────────
agg_df_01, full_df_01 = run_evaluation(
    experiment_name = "wound_ragas_eval_01_v2",
    results_json    = "../ragas_eval_01_v2/wound_ragas_ablation_results.json",
)
agg_df_01

# ──────────────────────────────────────────────────────────────────────────────



EXPERIMENT: wound_ragas_eval_01_v2
  [ 1/28] cat_a_type1_dry
       Safety: FAIL
  [ 2/28] cat_a_type2_wet
       Safety: FAIL
  [ 3/28] cat_a_type3_dry_infected
       Safety: FAIL
  [ 4/28] cat_a_type4_wet_infected
       Safety: PASS
  [ 5/28] cat_a_type5_dry_necrotic
       Safety: FAIL
  [ 6/28] cat_a_type6_wet_necrotic
       Safety: PASS
  [ 7/28] cat_a_type7_dry_infected_necrotic
       Safety: PASS
  [ 8/28] cat_a_type8_wet_infected_necrotic
       Safety: PASS
  [ 9/28] cat_b_iodine_thyroid
       Safety: FAIL
  [10/28] cat_b_silver_clean_granulating
       Safety: FAIL
  [11/28] cat_b_skin_tear_fragile
       Safety: FAIL
  [12/28] cat_b_npwt_necrotic_eschar
       Safety: FAIL
  [13/28] cat_b_alginate_dry_wound
       Safety: FAIL
  [14/28] cat_b_honey_dry_necrotic
       Safety: FAIL
  [15/28] cat_b_postop_clean
       Safety: PASS
  [16/28] cat_b_burns_hand
       Safety: FAIL
  [17/28] cat_b_referral_type6
       Safety: PASS
  [18/28] cat_b_diabetic_foot
       Safety:

Evaluating:  36%|███▌      | 40/112 [03:04<05:15,  4.38s/it]Exception raised in Job[12]: TimeoutError()
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating: 100%|██████████| 112/112 [08:16<00:00,  4.43s/it]



────────────────────────────────────────
  RAGAS SCORES — wound_ragas_eval_01_v2
────────────────────────────────────────
  Context precision         0.8632  (86.3%)
  Context recall            0.7196  (72.0%)
  Faithfulness              0.7217  (72.2%)
  Answer relevancy          0.5776  (57.8%)
────────────────────────────────────────
   Chart saved → ../ragas_eval_01_v2/wound_ragas_ablation_results_chart.png


,metric,score
0,Context precision,0.863210
1,Context recall,0.719641
2,Faithfulness,0.721727
3,Answer relevancy,0.577594


In [5]:
# ──────────────────────────────────────────────────────────────────────────────
# Cell 10: Eval 02
# ──────────────────────────────────────────────────────────────────────────────
agg_df_02, full_df_02 = run_evaluation(
    experiment_name = "wound_ragas_eval_02_v2",
    results_json    = "../ragas_eval_02_v2/wound_ragas_ablation_results.json",
)
agg_df_02




EXPERIMENT: wound_ragas_eval_02_v2
  [ 1/28] cat_a_type1_dry
       Safety: FAIL
  [ 2/28] cat_a_type2_wet
       Safety: FAIL
  [ 3/28] cat_a_type3_dry_infected
       Safety: PASS
  [ 4/28] cat_a_type4_wet_infected
       Safety: PASS
  [ 5/28] cat_a_type5_dry_necrotic
       Safety: FAIL
  [ 6/28] cat_a_type6_wet_necrotic
       Safety: PASS
  [ 7/28] cat_a_type7_dry_infected_necrotic
       Safety: PASS
  [ 8/28] cat_a_type8_wet_infected_necrotic
       Safety: PASS
  [ 9/28] cat_b_iodine_thyroid
       Safety: FAIL
  [10/28] cat_b_silver_clean_granulating
       Safety: FAIL
  [11/28] cat_b_skin_tear_fragile
       Safety: FAIL
  [12/28] cat_b_npwt_necrotic_eschar
       Safety: FAIL
  [13/28] cat_b_alginate_dry_wound
       Safety: FAIL
  [14/28] cat_b_honey_dry_necrotic
       Safety: FAIL
  [15/28] cat_b_postop_clean
       Safety: PASS
  [16/28] cat_b_burns_hand
       Safety: FAIL
  [17/28] cat_b_referral_type6
       Safety: PASS
  [18/28] cat_b_diabetic_foot
       Safety:

Evaluating:  38%|███▊      | 43/112 [03:02<03:38,  3.17s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating: 100%|██████████| 112/112 [07:54<00:00,  4.23s/it]



────────────────────────────────────────
  RAGAS SCORES — wound_ragas_eval_02_v2
────────────────────────────────────────
  Context precision         0.8044  (80.4%)
  Context recall            0.5159  (51.6%)
  Faithfulness              0.6760  (67.6%)
  Answer relevancy          0.5837  (58.4%)
────────────────────────────────────────
   Chart saved → ../ragas_eval_02_v2/wound_ragas_ablation_results_chart.png


,metric,score
0,Context precision,0.804444
1,Context recall,0.515909
2,Faithfulness,0.675954
3,Answer relevancy,0.583731


In [6]:
# ──────────────────────────────────────────────────────────────────────────────
# Cell 11: Comparison table
# ──────────────────────────────────────────────────────────────────────────────
comparison = pd.DataFrame({
    "Metric": METRIC_LABELS,
    "Eval_00 (baseline)":   [agg_df_00[agg_df_00.metric == lbl].score.values[0] for lbl in METRIC_LABELS],
    "Eval_01 (+BM25)":      [agg_df_01[agg_df_01.metric == lbl].score.values[0] for lbl in METRIC_LABELS],
    "Eval_02 (+reranker)":  [agg_df_02[agg_df_02.metric == lbl].score.values[0] for lbl in METRIC_LABELS],
})
comparison = comparison.set_index("Metric")
print(comparison.to_string())

                   Eval_00 (baseline)  Eval_01 (+BM25)  Eval_02 (+reranker)
Metric                                                                     
Context precision            0.866349         0.863210             0.804444
Context recall               0.658214         0.719641             0.515909
Faithfulness                 0.612632         0.721727             0.675954
Answer relevancy             0.573245         0.577594             0.583731
